# AdaptiveDepth Phase 7 - calibrated risk router
This CPU-only notebook trains the conservative prompt risk router from `phase6_artifacts.zip`. It does not load the LLM and does not require a GPU.

In [ ]:
import base64, os, pathlib, shutil, subprocess, sys, zipfile

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to Colab Secrets.')
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[router]'], check=True)
print('Repository:', repo)

## Mount Drive and extract Phase 6
The expected input is `MyDrive/AdaptiveDepth-results/phase6_artifacts.zip`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
archive = drive_results / 'phase6_artifacts.zip'
extract_root = pathlib.Path('/content/phase6_for_router')
if not archive.is_file():
    raise FileNotFoundError(f'Missing Phase 6 archive: {archive}')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(extract_root)
datasets = list(extract_root.rglob('router_dataset.jsonl'))
if len(datasets) != 1:
    raise RuntimeError(f'Expected one router dataset, found: {datasets}')
dataset = datasets[0]
phase6_gate = dataset.parent / 'gate.json'
if not phase6_gate.is_file():
    raise FileNotFoundError(f'Missing Phase 6 gate next to dataset: {phase6_gate}')
print('Dataset:', dataset)
print('Examples:', sum(1 for line in dataset.open(encoding='utf-8') if line.strip()))

In [ ]:
workspace = pathlib.Path('experiments/phase7')
subprocess.run([
    'adaptivedepth', 'train-phase7',
    '--dataset', str(dataset),
    '--phase6-gate', str(phase6_gate),
    '--config', 'configs/phase7.yaml',
    '--output', str(workspace),
], check=True)
print((workspace / 'gate.md').read_text())

In [ ]:
output_zip = pathlib.Path('/content/phase7_artifacts.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in workspace.rglob('*'):
        if path.is_file():
            bundle.write(path, workspace.name + '/' + str(path.relative_to(workspace)))
drive_results.mkdir(parents=True, exist_ok=True)
saved = shutil.copy2(output_zip, drive_results / output_zip.name)
shutil.copy2(workspace / 'gate.md', drive_results / 'phase7_gate.md')
shutil.copy2(workspace / 'report.json', drive_results / 'phase7_report.json')
print('Saved:', saved)
print('A revise_router verdict is a safe result; do not enable that artifact in Phase 8.')